# 1 物理实验与计算机实验

> **图 1.1**：房间高度与热烟层到达火源上方 5 英尺处所需时间的散点图

In [ ]:
if(!dir.exists("data"))dir.create("data")
if(!file.exists("data/1.1-plot.RData")){
  CT=1;CL=1;G=32.2;CP=0.24;PA=530.0;DA=0.075;LR=0.35;Q0=0.1
  targetZ=5
  QA_peak=1000;tpk=180;TMAX=1200
  QAt=function(t){if(t<=tpk)QA_peak*t/tpk else QA_peak} # 热释放速率
  QTfun=function(t)QAt(t)/Q0 # 热释放速率（归一化）
  asetb_orig=function(A,H,F,LC){ # 火灾物理模型
    Z0=H-F
    c1=(1-LC)*Q0*CT/(DA*CP*PA*A*CL)
    c2=(0.21*CT/A)*((1-LR)*Q0*G*CL^2/(DA*CP*PA))^(1/3)
    P0=1+2^(-5/3)*c1/c2
    DQ0=QA_peak/tpk/Q0
    deriv=function(Z,P,t){
      qt=QTfun(t);qt13=qt^(1/3)
      dZ=-c1*qt-c2*qt13*Z^(5/3)
      if(Z>=Z0-1e-6){
        dP=(c1/c2)*(2*DQ0+5*(c1+c2*Z0^(5/3)))/(6*Z0^(8/3))
      }else{
        dP=P*(c1*qt-(P-1)*c2*qt13*Z^(5/3))/(Z0-Z)
      }
      c(dZ,dP)
    }
    dt=1.0;Z=Z0;P=P0;t=0
    Zv=c(Z0);tv=c(0)
    while(t<TMAX){ # Heun 法数值积分
      d1=deriv(Z,P,t)
      Zp=Z+d1[1]*dt;Pp=P+d1[2]*dt
      d2=deriv(Zp,Pp,t+dt)
      Z=Z+0.5*(d1[1]+d2[1])*dt
      P=P+0.5*(d1[2]+d2[2])*dt
      t=t+dt
      if(Z<0)Z=0
      Zv=c(Zv,Z);tv=c(tv,t)
      if(Z<=targetZ)break
    }
    if(min(Zv)>targetZ)return(NA)
    idx=which(Zv<=targetZ)[1]
    if(idx==1)return(0)
    t1=tv[idx-1];t2=tv[idx];z1=Zv[idx-1];z2=Zv[idx]
    if(z2==z1)return(t2)
    t1+(targetZ-z1)*(t2-t1)/(z2-z1)
  }
  x_lower=c(LC=0.6,F=1.0,H=8.0,A=81.0)
  x_upper=c(LC=0.9,F=3.0,H=12.0,A=256.0)
  sobol_m=function(s,a,m0,bits=30L){ # Sobol 序列
    if(s==0L)return(rep(1L,bits))
    m=integer(bits)
    m[1:s]=m0
    for(k in(s+1L):bits){
      v=bitwXor(bitwShiftL(m[k-s],s),m[k-s])
      if(s>1L)for(i in 1:(s-1L))
        if(a[i]==1L)v=bitwXor(v,bitwShiftL(m[k-i],i))
      m[k]=v
    }
    m
  }
  my_sobol=function(n,dim=4L,B=30L){ # 把方向数变成二进制小数
    poly=list(
      list(0L,integer(0),1L),
      list(1L,integer(0),1L),
      list(2L,1L,c(1L,1L)),
      list(3L,c(0L,1L),c(1L,3L,7L)))
    D=matrix(0L,B,dim)
    for(j in 1:dim){
      m=sobol_m(poly[[j]][[1]],poly[[j]][[2]],poly[[j]][[3]],B)
      D[,j]=bitwShiftL(m,B -(1:B))
    }
    x=integer(dim)
    out=matrix(0L,n,dim)
    for(i in 1:n){
      t=i-1L
      c=0L
      while(bitwAnd(t,1L)==1L){ t=bitwShiftR(t,1L); c=c+1L }
      x=bitwXor(x,D[c+1L,])
      out[i,]=x
    }
    out/2^B
  }
  n=40
  Xu=my_sobol(n=n+1,dim=4)[-1,]
  X=t(apply(Xu,1,function(r)x_lower+r*(x_upper-x_lower))) # 生成 40 个 (x,y)
  colnames(X)=names(x_lower)
  y=numeric(n)
  for(i in 1:n){y[i]=asetb_orig(A=X[i,"A"],H=X[i,"H"],F=X[i,"F"],LC=X[i,"LC"])}
  df=data.frame(X,y=y)
  save(asetb_orig,df,file="data/1.1-plot.RData")
}
load("data/1.1-plot.RData")
xl=c(LC=0.6,F=1,H=8,A=81);xu=c(LC=0.9,F=3,H=12,A=256)
Xu0=t(apply(as.matrix(df[,1:4]),1,function(r)(r-xl)/(xu-xl)))
dmin=Inf
for(i in 1:(nrow(Xu0)-1))for(j in (i+1):nrow(Xu0))dmin=min(dmin,sqrt(sum((Xu0[i,]-Xu0[j,])^2)))
cat("最小点间距离:",round(dmin,4),"\n")
library(svglite)
svglite("../markdown/figures/1.1.svg",width=4,height=4)
plot(df$H,df$y,pch=16,cex=1,xlab="房间高度（英尺）",ylab="到达 5 英尺所需时间（秒）")
dev.off()

最小点间距离: 0.2073 


agg_record_48443b0740df 
                      2